In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/MMVision"

Path(f"{BASE_DIR}/models/captioning").mkdir(parents=True, exist_ok=True)

In [9]:
%%writefile /content/drive/MyDrive/MMVision/models/captioning/c1.py

import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
import torchvision.models as tv_models
import torch.nn.functional as F


device = "cuda" if torch.cuda.is_available() else "cpu"
# -----------------------------
# Encoder
# -----------------------------
class ResnetImageEncoder(nn.Module):

    def __init__(self, embed_dim=512):
        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2
        )

        # Freeze initially
        for param in backbone.parameters():
            param.requires_grad = False

        # Fine-tune only deeper layers
        for param in backbone.layer4.parameters():
            param.requires_grad = True

        self.backbone = nn.Sequential(
            *list(backbone.children())[:-2]
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.projection = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, embed_dim),
            nn.LayerNorm(embed_dim)
        )

        for m in self.projection:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, images):

        features = self.backbone(images)

        # (B,2048,7,7)

        features = self.pool(features)

        # (B,2048,1,1)

        features = torch.flatten(features, 1)

        # (B,2048)

        features = self.projection(features)

        # (B,512)

        return features
# -----------------------------
# Decoder
# -----------------------------
class LSTMCaptionDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        hidden_dim=512,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(0.3)

        # Initialize LSTM from image feature
        self.init_h = nn.Linear(embed_dim, hidden_dim)
        self.init_c = nn.Linear(embed_dim, hidden_dim)

        self.h_norm = nn.LayerNorm(hidden_dim)
        self.c_norm = nn.LayerNorm(hidden_dim)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.output_dropout = nn.Dropout(0.3)

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, vocab_size)
        )

        # -----------------------
        # Initialization
        # -----------------------

        nn.init.xavier_uniform_(self.embedding.weight)

        with torch.no_grad():
            self.embedding.weight[pad_idx].fill_(0)

        for m in self.fc:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

        nn.init.xavier_uniform_(self.init_h.weight)
        nn.init.zeros_(self.init_h.bias)

        nn.init.xavier_uniform_(self.init_c.weight)
        nn.init.zeros_(self.init_c.bias)

        for name, param in self.lstm.named_parameters():

            if "weight_ih" in name:
                nn.init.xavier_uniform_(param)

            elif "weight_hh" in name:
                nn.init.orthogonal_(param)

            elif "bias" in name:
                nn.init.zeros_(param)

                # Forget gate bias = 1
                hidden = param.size(0) // 4
                param.data[hidden:2*hidden].fill_(1)

    def forward(
        self,
        image_features,
        captions
    ):

        embeddings = self.embedding(captions)

        embeddings = self.embedding_dropout(embeddings)

        h0 = torch.tanh(
            self.h_norm(
                self.init_h(image_features)
            )
        ).unsqueeze(0)

        c0 = torch.tanh(
            self.c_norm(
                self.init_c(image_features)
            )
        ).unsqueeze(0)

        outputs, _ = self.lstm(
            embeddings,
            (h0, c0)
        )

        outputs = self.output_dropout(outputs)

        outputs = self.fc(outputs)

        return outputs
# -----------------------------
# Joint Model
# -----------------------------

class ResnetLSTMImageCaptioningModel(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, captions):

        image_features = self.encoder(images)

        outputs = self.decoder(
            image_features,
            captions
        )

        return outputs

    @staticmethod
    def length_penalty(length, alpha=0.7):
        return ((5 + length) ** alpha) / ((5 + 1) ** alpha)

    def generate_caption(
        self,
        images,
        vocab,
        max_length=30,
        beam_size=5
    ):

        self.eval()

        batch_size = images.size(0)

        assert batch_size == 1, \
            "Beam search currently supports batch_size=1."

        with torch.no_grad():

            image_features = self.encoder(images)

            h = torch.tanh(
                self.decoder.h_norm(
                    self.decoder.init_h(image_features)
                )
            ).unsqueeze(0)

            c = torch.tanh(
                self.decoder.c_norm(
                    self.decoder.init_c(image_features)
                )
            ).unsqueeze(0)

            beams = [
                (
                    [vocab["<SOS>"]],
                    0.0,
                    (h, c)
                )
            ]

            completed = []

            for step in range(max_length):

                candidates = []

                for seq, score, hidden in beams:

                    last_token = seq[-1]

                    if last_token == vocab["<EOS>"]:
                        completed.append((seq, score))
                        continue

                    current_word = torch.tensor(
                        [last_token],
                        device=images.device
                    )

                    embedding = self.decoder.embedding(
                        current_word
                    )

                    embedding = self.decoder.embedding_dropout(
                        embedding
                    )

                    output, hidden_new = self.decoder.lstm(
                        embedding.unsqueeze(1),
                        hidden
                    )

                    scores = self.decoder.fc(
                        self.decoder.output_dropout(
                            output.squeeze(1)
                        )
                    )

                    # Block unwanted tokens
                    scores[:, vocab["<PAD>"]] = -1e9
                    scores[:, vocab["<SOS>"]] = -1e9

                    if "<UNK>" in vocab:
                        scores[:, vocab["<UNK>"]] = -1e9

                    # Prevent early EOS
                    if step < 3:
                        scores[:, vocab["<EOS>"]] = -1e9

                    log_probs = F.log_softmax(
                        scores,
                        dim=1
                    )

                    top_log_probs, top_indices = torch.topk(
                        log_probs,
                        beam_size,
                        dim=1
                    )

                    for k in range(beam_size):

                        token = top_indices[0, k].item()

                        new_seq = seq + [token]

                        new_score = score + \
                            top_log_probs[0, k].item()

                        # Clone hidden state
                        h_new = hidden_new[0].clone()
                        c_new = hidden_new[1].clone()

                        candidates.append(
                            (
                                new_seq,
                                new_score,
                                (h_new, c_new)
                            )
                        )

                if len(candidates) == 0:
                    break

                candidates = sorted(
                    candidates,
                    key=lambda x:
                        x[1] /
                        self.length_penalty(len(x[0])),
                    reverse=True
                )

                beams = candidates[:beam_size]

            if len(completed) == 0:
                completed = [
                    (
                        beams[0][0],
                        beams[0][1]
                    )
                ]

            completed = sorted(
                completed,
                key=lambda x:
                    x[1] /
                    self.length_penalty(len(x[0])),
                reverse=True
            )

            best_seq = completed[0][0]

            return [best_seq]
# -----------------------------
#image transform
# -----------------------------

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# -----------------------------
# Predict Caption
# -----------------------------

def caption_from_image(
    model,
    image,
    vocab,
    max_length=30
):

    model.eval()

    device = next(model.parameters()).device

    image = image_transform(image).unsqueeze(0).to(device)

    idx2word = {v: k for k, v in vocab.items()}

    with torch.no_grad():

        ids = model.generate_caption(
            image,
            vocab,
            max_length=max_length
        )[0]

    words = []

    for idx in ids:

        word = idx2word.get(idx, "<UNK>")

        if word == "<EOS>":
            break

        if word not in ["<SOS>", "<PAD>","<UNK>"]:
            words.append(word)

    return " ".join(words)
# -----------------------------
# Build Model
# -----------------------------
def build_C1(vocab):

  encoder = ResnetImageEncoder(embed_dim=512)

  decoder = LSTMCaptionDecoder(
      vocab_size=len(vocab),
      embed_dim=512,
      hidden_dim=512,
      pad_idx=vocab["<PAD>"]
  )

  model = ResnetLSTMImageCaptioningModel(
      encoder,
      decoder
  ).to(device)

  return model

Overwriting /content/drive/MyDrive/MMVision/models/captioning/c1.py
